# Sessions 14–15: Sequence Models and Transformer Applications

**Learning goals:**

- Trace raw text through tokenization, embeddings, contextual representations and task outputs.
- Explain RNN and LSTM state, attention and Transformer information flow using examples.
- Distinguish sequence-to-sequence generation from sequence classification.
- Compare encoder–decoder, encoder-only and decoder-only language models.
- Fine-tune DistilBERT for support ticket classification.
- Compare held-out results, analyse mistakes and save models for reuse.

**Prerequisites:**

- Neural networks, PyTorch, training, evaluation and transfer learning.
- Basic Python functions, lists, dictionaries and tensor shapes.
- Internet for initial model and dataset downloads.



## Outline

1. From CNNs to sequence models
2. Tokenization and embedding
3. RNN and LSTM: a brief introduction
4. Attention and Transformers
5. Large language models for natural language processing
6. Case study: support ticket classification


## 1. From CNNs to Sequence Models

Deep learning learns useful **feature representations** from raw data, then uses them to produce a task output. “Deep” refers to multiple layers that progressively transform and combine information; training adjusts these layers and the output head together.

**Raw input → learned feature representations → task output**

For an image, early CNN layers can detect edges and simple patterns. Later layers combine them into more complex shapes and object features. A classification head uses these features to predict a class.

![CNN: image to tensor, learned spatial features, classification head and output.](assets/session14-15/cnn-image2.png)


**Sequence data** includes text, speech and time-series measurements. Its elements are ordered and often depend on one another:

- **Order matters:** “The dog chased the cat” differs from “The cat chased the dog”.
- **Context matters:** “bank” means something different beside “loan” and “river”.
- **Length varies:** sentences contain different numbers of tokens, and a translation can differ in length from its source.

Sequence models learn features from these relationships. In ticket classification, a Transformer combines information across the text, and a classification head predicts the ticket category.

![Sequence model: text to token IDs, embeddings, contextual features, classification head and output.](assets/session14-15/sequence-image2.png)



## 2. Tokenization and Embedding

Text needs two conversions before the network processes it: **tokenization** splits text into tokens and assigns IDs; **embedding** turns those IDs into learned numerical vectors. Transformer layers then combine information across positions to build contextual features.

![Text passes through a tokenizer into IDs, embedding vectors and Transformer layers to produce contextual features.](assets/session14-15/text-model-overview-image2.png)


### 2.1 Tokenization

A **tokenizer** splits text into units such as words, subword pieces or characters, then looks up each unit in its vocabulary. Tokens can have different granularities:

- **Character-level:** each token represents a character.
- **Word-level:** each token represents a whole word.
- **Subword-level:** each token represents a subword piece; a word may use one or several pieces.

For example, an illustrative split of `cats` is `c | a | t | s` at character level, `cats` at word level, or `cat | s` at subword level. Actual subword boundaries depend on the tokenizer.

For the toy vocabulary below, `Cats sleep.` becomes `[12, 47, 3]`.

![A toy tokenizer splits Cats sleep. into pieces and maps them to vocabulary IDs.](assets/session14-15/tokenization-image2.png)

**The tokenizer is paired with the model.** Models using different tokenizers can produce different token boundaries, IDs and token counts for the same text. Some models share a tokenizer, so their results can match. Always use the tokenizer supplied for your model: an ID must select the embedding row that the model learned for that token.

**Explore:** Open the [OpenAI Tokenizer](https://platform.openai.com/tokenizer) and enter `An unpredictable Wi-Fi problem!`. Inspect the pieces and token count. Where model or encoding choices are available, compare the same text across them; try adding a space or changing a word. Do not assume a token equals one word or character.


### 2.2 Embedding

**Purpose:** token IDs identify vocabulary entries but do not describe their meaning. Embeddings give the network compact numerical features it can learn to use. Unlike a long, sparse one-hot vector, an embedding contains a smaller set of continuous values.

**Method:** embedding converts each token ID into a fixed-dimensional vector. The model stores an **embedding table**, with one vector per vocabulary entry. These vectors are usually **model parameters learned through training**: embedding gives each token a **learnable vector**. Each token ID selects its row. The selected vectors stay in input order: three tokens with four values each form a `3 × 4` matrix.

![Token IDs select learned vectors in input order from an embedding table.](assets/session14-15/embedding-image2.png)

The table values are learned during pretraining and can change during fine-tuning. Training can give related tokens useful shared patterns; nearby ID numbers do not imply similar meanings, and vector dimensions have no fixed human labels.

Embedding is the starting representation. For the same `bank` token ID, the initial embedding is identical in `river bank` and `bank loan`. After Transformer layers combine context and position information, the **contextual representations differ**, reflecting the different surrounding words. These contextual features support classification or generation.


## 3. RNN and LSTM: A Brief Introduction

A **Recurrent Neural Network (RNN)** reads a sequence one step at a time. Each step combines the current input with the previous hidden state to produce a new representation. The same parameters are reused across steps.

A **Long Short-Term Memory (LSTM)** network adds a memory pathway and gates to control what information is kept, updated and exposed. This helps it learn longer dependencies.


### A Character-Level RNN Example

![An RNN reads h, e, l, l and scores the next characters e, l, l, o.](assets/session14-15/rnn-hell-forward-pass.png)

Read the figure from left to right:

- **Input:** `h, e, l, l`, encoded with four one-hot values for vocabulary `h, e, l, o`.
- **Hidden state:** three units carry context between steps.
- **Output:** four next-character scores; targets are `e, l, l, o`. Training raises green target scores relative to red alternatives. These are raw scores, not probabilities.

The two `l` inputs have different targets: the hidden state lets the model use their different contexts.

[Karpathy: The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/) · [Awesome RNN resources](https://github.com/kjw0612/awesome-rnn)

### LSTM: Managing Memory Across Time

![Two LSTM steps use gates to update cell and hidden states.](assets/session14-15/lstm-memory-gates-image2.png)

The **cell state** is the upper memory pathway. The **hidden state** is the current output representation passed to the next step.

| Gate | Role |
|---|---|
| Forget | Retain useful old memory. |
| Input | Write selected new information. |
| Output | Control how much updated memory contributes to the hidden state. |

Gates use continuous values between 0 and 1. For “I grew up in France … I speak fluent French”, useful information about France can be retained across intervening words. Memory remains limited, and processing is still sequential.

[Hochreiter and Schmidhuber: Long Short-Term Memory (1997)](https://doi.org/10.1162/neco.1997.9.8.1735)


Imagine translating a long notice containing a date, room number and several instructions. A single final summary must support all output decisions. This fixed-size bottleneck motivates giving the decoder access to representations at all input positions.



## 4. Attention and Transformers

### 4.1 Attention

**Attention builds a useful feature vector by giving more weight to relevant information and less weight to other information.** In RNN/LSTM Seq2Seq models, it lets the decoder access different input positions rather than depend only on a final summary.

**Input → output:**

| Input | Role |
|---|---|
| Query (Q) | The current representation requesting information. |
| Keys (K) | Representations compared with the query to calculate weights. |
| Values (V) | Information combined using those weights. |

**Query + Keys → attention weights → weighted combination of Values → output feature vector.** Each query produces an output feature vector. Attention does not directly output a translated word; later decoder processing and the vocabulary output layer predict it.

![Attention principle: a query is compared with keys to calculate weights; the weights combine values into an output feature vector.](assets/session14-15/attention-principle-image2.png)

**Read the attention diagram:** thicker arrows show a larger contribution from `nine`. The combined feature vector helps later layers predict `neuf` (“nine”). The lower flow shows how this happens: compare Q with K to calculate weights, then use those weights to combine V. The illustrated weights are not measurements from a trained model.

**Example: different output steps use different source weights**

![](assets/session14-15/attention-translation-example-image2.png)

Read each row from left to right. When predicting `J’aime`, the query gives weights `0.45, 0.50, 0.05` to the source positions `I, like, tea`. When predicting `thé`, the weights become `0.05, 0.05, 0.90`, focusing more strongly on `tea`. Each row sums to 1 and produces its own context vector by combining source value vectors.

The source representations stay available, but the decoder query and weights change with the generated prefix. Decoder processing and the output head use the context vector to help predict the next word. This is a word-level illustration with invented weights; an actual model operates on its tokenizer’s tokens.

**Multi-head attention**

![Several attention heads learn separate projections and weights; their output vectors are concatenated and projected to produce updated representations.](assets/session14-15/multihead-attention-image2.png)

The input representations enter several heads, each with its own learned projections and weights. Their output vectors are concatenated and projected to form updated features. Read the diagram from left to right: input → parallel heads → concatenate and project → updated representations.

In **self-attention**, Q, K and V come from the same sequence. In **cross-attention**, Q comes from the decoder while K and V come from encoder outputs, connecting output generation to the source.

[Translation attention](https://arxiv.org/abs/1409.0473) · [Multi-head attention](https://arxiv.org/abs/1706.03762)


### 4.2 Transformer

The original **Transformer** has three key structural features:

- **Seq2Seq structure:** an encoder represents the source sequence; a decoder uses those representations to generate a target sequence.
- **No recurrent state updates:** it removes the step-by-step recurrence used by RNNs/LSTMs, allowing many sequence positions to be calculated in parallel during training.
- **Repeated attention:** stacked blocks use multi-head attention and feed-forward layers to refine features. Position information makes sequence order available.

![](assets/session14-15/encoder-decoder-input-output-image2.png)

| Component | Input | Output |
|---|---|---|
| Encoder | Source token embeddings with position information | A sequence of contextual source features |
| Decoder | Source features and target-prefix embeddings with position information | Contextual target features |
| Vocabulary output head | Decoder features | Scores over possible next tokens |

The decoder therefore uses **two sources of information**: what the input says and what has already been generated. It produces feature vectors; the output head converts them into token scores.

**Example: “I like tea” → “J’aime le thé”**

![](assets/session14-15/encoder-decoder-translation-example-image2.png)

The encoder produces a contextual representation for each source position. At every generation step, the decoder can access those source features and uses an increasingly long target prefix:

| Step | Target prefix | Next output |
|---|---|---|
| 1 | `<start>` | `J’aime` |
| 2 | `<start> J’aime` | `le` |
| 3 | `<start> J’aime le` | `thé` |

The three decoder panels show the **same model reused across steps**, not three separately trained decoders. This word-level example omits the final end token and simplifies tokenization.

![](assets/session14-15/transformer-seq2seq-image2.png)

Encoder self-attention combines source information. Decoder masked self-attention uses the target prefix, and cross-attention accesses encoder features. Feed-forward layers further transform the representations.

**Translation at inference:** encode the source, start the decoder with a start token, predict a token, append it to the prefix and repeat until an end token or length limit.

During training, shifted reference targets supply the prefixes and a **causal mask** hides future target tokens. Multiple positions can be calculated in parallel; autoregressive generation still proceeds token by token.


## 5. Large Language Models for Natural Language Processing

Pretrained language models reuse Transformer representations across language tasks. Three common architecture families differ in how they read input and produce output. The examples include both early pretrained models and later large models.

| Architecture | Main role and applications | Examples |
|---|---|---|
| **Encoder–decoder** | Encode a source sequence, then generate a target sequence. Used for translation, summarisation and response generation. | Original Transformer; Google **T5** |
| **Encoder-only** | Build bidirectional contextual representations of input. Add a task head for classification, sentiment analysis, tagging or extractive question answering. | Google **BERT** |
| **Decoder-only** | Extend a prompt by predicting tokens from the available prefix. Used for continuation, dialogue and other prompted generation tasks. | OpenAI **GPT**, Meta **LLaMA/Llama**, **DeepSeek-V3/R1** |

Encoder-only describes the architecture, not one-way reading: BERT uses both left and right context. Decoder-only models have no separate source encoder; the prompt and generated tokens form their working sequence.

**GLM:** the original [GLM](https://arxiv.org/html/2103.10360v2) combines bidirectional context processing and autoregressive blank filling within a single Transformer. It supports understanding and generation, but is not a conventional two-stack encoder–decoder like T5; the architecture should be checked for the specific GLM version.

### 5.1 Selected Development Milestones

![Selected milestones from Transformer in 2017 to DeepSeek-R1 in 2025, grouped by architecture family.](assets/session14-15/nlp-model-timeline-horizontal-image2.png)

Read from left to right through time, from 2017 to 2025. The branches group architecture families; they do not show direct model ancestry or rank model quality. T5 first appeared as a 2019 preprint and was published in JMLR in 2020. This is a selection of milestones through 2025.

Timeline sources: [Transformer](https://arxiv.org/abs/1706.03762), [BERT](https://arxiv.org/abs/1810.04805), [T5](https://arxiv.org/abs/1910.10683), [GPT](https://openai.com/index/language-unsupervised/), [GPT-2](https://openai.com/index/better-language-models/), [GPT-3](https://arxiv.org/abs/2005.14165), [LLaMA](https://arxiv.org/abs/2302.13971), [Llama 3](https://ai.meta.com/blog/meta-llama-3/), [DeepSeek-V3](https://arxiv.org/abs/2412.19437), [DeepSeek-R1](https://arxiv.org/abs/2501.12948).

### 5.2 BERT: Representing Input with Both-Sided Context

**BERT** means **Bidirectional Encoder Representations from Transformers**. Its encoder uses left and right context to build a representation for each token.

- **Pretraining:** predict masked tokens; original BERT also learned a next-sentence prediction task.
- **Task use:** attach and fine-tune a task head, such as a ticket-category classifier.
- **Example:** `The bank approved the [MASK].` uses surrounding words to suggest the missing token.

The ticket practical uses **DistilBERT**, a smaller BERT-family encoder, with a new classification head fine-tuned on labelled tickets.


### 5.3 GPT: Generating from a Prefix

**GPT** means **Generative Pre-trained Transformer**. A decoder-only model uses causal self-attention: each position can use the available prefix, but not future tokens.

- **Pretraining:** predict the next token from preceding tokens.
- **Generation:** predict a token, append it to the prefix and repeat.
- **Applications:** text continuation, dialogue, summarisation and code generation, depending on the model and its subsequent training.

**Example:** `The support team checked the network and` becomes the prefix for continued text. Prompting changes the input; fine-tuning changes model parameters.

GPT-2 is a base continuation model; instruction-following assistants require further training. Generated text can contain repetition or incorrect claims.


## 6. Case Study: Support Ticket Classification

### 6.1 Define the Task and Inspect the Data

**Raw ticket → token IDs → embeddings → contextual encoder features → classification head → category.**

[Download the supplied ticket CSV](sample_data/support_tickets.csv).

The supplied CSV contains 80 fictional teaching tickets, with 20 in each class and explicit train/validation/test splits. It is deliberately small and balanced. It cannot estimate real service-desk performance, and short synthetic wording can make classification easier than realistic tickets.

| Category | Routing rule | Example |
|---|---|---|
| Account access | Credentials, locked accounts and authentication | My account is locked. |
| Hardware | Physical device faults | The laptop screen is cracked. |
| Software | Application installation, crashes and behaviour | The spreadsheet application crashes. |
| Network | Connectivity and network access | The campus Wi-Fi disconnects. |

A real ticket can involve several issues. This exercise assigns one primary category; ambiguous cases should be reviewed rather than forcing an unjustified label.


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

TICKET_CSV = "sample_data/support_tickets.csv"
tickets = pd.read_csv(TICKET_CSV)  # Open the supplied labelled data when first used.
print(tickets.groupby(["split", "category"]).size().unstack(fill_value=0))
print(tickets.head(8).to_string(index=False))
assert not tickets["text"].str.lower().duplicated().any(), "Check duplicate ticket text before training."

class_names = ["Account access", "Hardware", "Software", "Network"]
label_to_id = {name: number for number, name in enumerate(class_names)}
id_to_label = {number: name for name, number in label_to_id.items()}
tickets["label"] = tickets["category"].map(label_to_id)
ticket_splits = DatasetDict({
    name: Dataset.from_pandas(tickets.loc[tickets["split"] == name, ["text", "label"]], preserve_index=False)
    for name in ["train", "validation", "test"]
})


### 6.2 Prepare Inputs and Add the Classification Head

Each ticket label is one integer identifying its category. DistilBERT supplies pretrained language features, and a newly initialised classification head supplies four task scores. The initialisation warning for the new head is expected: its categories have not been learned yet.

Padding makes batch lengths compatible; its attention mask marks real tokens. Truncation limits input length and can remove useful details from long tickets.


Before using Transformers, install these packages in your terminal, then restart the notebook kernel. The practical uses the Transformers 4.57 API. Run from the course folder; a GPU is recommended for training.

```bash
python -m pip install "torch>=2.6,<3" "transformers==4.57.1" "datasets>=3,<5" "accelerate>=1,<2" scikit-learn pandas matplotlib
```


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed

BERT_CHECKPOINT = "distilbert/distilbert-base-uncased"
TICKET_TOKENIZER_CACHE_DIR = "../archive/AI_SS/session14-15/ticket-tokenizer-cache"
# Use the tokenizer paired with the pretrained encoder.
ticket_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT, cache_dir=TICKET_TOKENIZER_CACHE_DIR)
SEED = 42

MAX_TICKET_TOKENS = 96

def prepare_tickets(batch):
    return ticket_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

encoded_tickets = ticket_splits.map(prepare_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Initialise the new task head reproducibly.
TICKET_MODEL_CACHE_DIR = "../archive/AI_SS/session14-15/ticket-model-cache"
ticket_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT, num_labels=len(class_names), cache_dir=TICKET_MODEL_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
)
ticket_collator = DataCollatorWithPadding(tokenizer=ticket_tokenizer)


### 6.3 Train and Validate

Accuracy counts all correct predictions. Macro F1 gives each category equal weight; a confusion matrix shows where errors occur. Both the encoder and classification head update here. Freezing the encoder would instead train a task head on fixed language features.

Compare against a majority-class baseline. The untrained head is a useful diagnostic but is not an already competent ticket classifier.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import Trainer, TrainingArguments

def ticket_metrics(prediction):
    logits, labels = prediction
    predicted = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(labels, predicted, labels=list(id_to_label), average="macro", zero_division=0),
    }

TICKET_CHECKPOINT_DIR = "../archive/AI_SS/session14-15/ticket-checkpoints"
ticket_args = TrainingArguments(
    output_dir=TICKET_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=3, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
ticket_trainer = Trainer(
    model=ticket_model, args=ticket_args,
    train_dataset=encoded_tickets["train"], eval_dataset=encoded_tickets["validation"],
    processing_class=ticket_tokenizer, data_collator=ticket_collator, compute_metrics=ticket_metrics,
)
before_ticket = ticket_trainer.predict(encoded_tickets["test"])
ticket_trainer.train()
print("Validation:", ticket_trainer.evaluate())


### 6.4 Evaluate the Held-Out Tickets

Choose the checkpoint using validation results, then inspect test performance. The test set contains only 16 tickets; one changed prediction can substantially change the metrics. Review examples as well as aggregate scores.


In [ ]:
import matplotlib.pyplot as plt

after_ticket = ticket_trainer.predict(encoded_tickets["test"])
true_labels = after_ticket.label_ids
predicted_labels = after_ticket.predictions.argmax(axis=-1)
majority_label = int(tickets.loc[tickets["split"] == "train", "label"].mode().iloc[0])
majority_predictions = [majority_label] * len(true_labels)
print("Majority baseline accuracy:", accuracy_score(true_labels, majority_predictions))
print("Before:", ticket_metrics((before_ticket.predictions, true_labels)))
print("After:", ticket_metrics((after_ticket.predictions, true_labels)))
print(classification_report(
    true_labels, predicted_labels, labels=list(id_to_label), target_names=class_names, zero_division=0,
))
ConfusionMatrixDisplay.from_predictions(
    true_labels, predicted_labels, labels=list(id_to_label), display_labels=class_names,
    xticks_rotation=30, cmap="Blues", colorbar=False,
)
plt.tight_layout()
plt.show()
ticket_results = pd.DataFrame({
    "text": ticket_splits["test"]["text"],
    "actual": [id_to_label[int(label)] for label in true_labels],
    "predicted": [id_to_label[int(label)] for label in predicted_labels],
})
print("Errors:")
print(ticket_results[ticket_results["actual"] != ticket_results["predicted"]].to_string(index=False))


### 6.5 Save, Reload and Route New Tickets

A softmax score is relative to the model's available classes. It is not a calibrated probability that the routing decision is correct. Out-of-scope tickets and ambiguous requests need a review process.


In [ ]:
from transformers import pipeline

TICKET_SAVE_DIR = "../archive/AI_SS/session14-15/saved-ticket-classifier"
ticket_trainer.save_model(TICKET_SAVE_DIR)
ticket_tokenizer.save_pretrained(TICKET_SAVE_DIR)
ticket_classifier = pipeline("text-classification", model=TICKET_SAVE_DIR, tokenizer=TICKET_SAVE_DIR)
new_tickets = [
    "My laptop cannot connect to campus Wi-Fi.",
    "I dropped my monitor and the display is broken.",
    "The program crashes, but only when I connect to Wi-Fi.",
]
for text, result in zip(new_tickets, ticket_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)):
    print(text, "→", result)
